# 03b — Experimentos decisivos: conflicto vs. pobreza/región
## Fase 3 del TFM — reproducción de los números que sostienen el reencuadre del proyecto

**Contexto (para que este notebook se entienda sin tener que abrir nada más):** el diseño
original de este proyecto planteaba que los residuos de un modelo estructural (NBI + intensidad
de conflicto armado, `per_ocu`) revelarían anomalías de integridad electoral en zonas de
conflicto — una hipótesis de coacción territorial. Un primer experimento (no incluido en este
notebook, ver `03_modelizacion.ipynb` y la bitácora de Fase 3) mostró que los municipios de zonas
de conflicto sí votan de forma distinta a lo que un modelo con inercia electoral predice —una
brecha de hasta +13/+38 puntos según el año—. Pero esa brecha, por sí sola, no distingue entre dos
explicaciones muy distintas: (a) el conflicto armado tiene un efecto propio sobre el voto, o
(b) las zonas de conflicto son también las más pobres y periféricas de Colombia, y lo que se
observa es en realidad el efecto de la pobreza y la región, no del conflicto.

Este notebook resuelve esa ambigüedad con dos comprobaciones, reproducibles de principio a fin:

1. **Experimento 2 (decisivo)**: ¿aporta `per_ocu` capacidad explicativa sobre el voto de
   izquierda por encima de lo que ya explican `nbi_total` (pobreza) y `region_dane` (proxy de
   periferia)? Si la respuesta es "casi nada", la explicación (b) gana sobre la (a).
2. **Comprobación de robustez de un hallazgo lateral**: al hacer la comprobación anterior año a
   año, apareció un patrón llamativo (el R² de NBI+región crece mucho entre 2014 y 2022) que
   podría ser un segundo hallazgo real ("el voto se estructuró crecientemente") o un artefacto
   estadístico (más varianza disponible para explicar en unos años que en otros). Se pone a
   prueba con cuatro comprobaciones antes de aceptarlo o descartarlo.

Ambas comprobaciones fueron las que decidieron reencuadrar el proyecto de "detección de coacción
electoral" a "estudio analítico del comportamiento electoral territorial" (ver prompt de proyecto
v9 y memoria final). Detalle narrativo completo del recorrido, incluida la ronda de experimentos
previa: bitácora interna de Fase 3 (`bitacora_fase3_modelizacion_v*.md`, §6-§8, no versionada en
git) — pero los números que sostienen la conclusión están todos aquí, ejecutados.

In [1]:
import sys
sys.path.insert(0, '../src')
import pandas as pd
import numpy as np
import modelo as m

pd.set_option('display.width', 140)

BASE_DIR = '..'
df = pd.read_csv(f'{BASE_DIR}/datos/procesados/dataset_maestro_electoral.csv')
df = df[df['valido_para_modelado'] == 1].copy()

# Respetando las banderas de confiabilidad, como en el resto del pipeline formal:
df_conf = df[df['baja_confiabilidad_electoral'] == 0].copy()
print(f'Filas validas para modelado: {len(df)}')
print(f'Filas tras excluir baja_confiabilidad_electoral=1: {len(df_conf)} '
      f'(excluidas {len(df) - len(df_conf)})')
print(f'\nDepartamentos de zona de conflicto (m.ZONA_CONFLICTO_DEPARTAMENTOS): {m.ZONA_CONFLICTO_DEPARTAMENTOS}')

Filas validas para modelado: 5604
Filas tras excluir baja_confiabilidad_electoral=1: 5584 (excluidas 20)

Departamentos de zona de conflicto (m.ZONA_CONFLICTO_DEPARTAMENTOS): ['CHOCO', 'PUTUMAYO', 'NARINO', 'CAQUETA', 'NORTE DE SANTANDER', 'GUAVIARE', 'VAUPES']


## 1. Experimento 2 (decisivo) — ¿aporta `per_ocu` más allá de NBI+región?

Dentro de cada año por separado (2014, 2018, 2022), cross-seccional, regresión ponderada por
`peso_muestral` (in-sample: aquí el objetivo es explicativo, no predictivo — se compara cuánta
varianza del voto explican los controles frente a cuánta explican controles+conflicto, dentro del
propio año, no la capacidad de generalizar a otro año).

- **Control**: `nbi_total` + `region_dane` (dummies) — proxy de ruralidad/pobreza/periferia. No
  hay proxy de ruralidad/población en el dataset actual; se usa `region_dane` como aproximación,
  explícitamente no como control fino (las zonas de conflicto están concentradas en
  Amazonía/Orinoquía/Pacífico).
- **Prueba**: ΔR² al añadir `per_ocu` al control, y correlación parcial de `per_ocu` con el target
  controlando por NBI+región (vía residualización: se resta a cada variable lo que predicen los
  controles, y se correlacionan los residuos).
- **Contraste adicional**: la brecha de residuo (target ya controlado por NBI+región) entre
  zona_conflicto y el resto — para ver si sigue existiendo un efecto propio de conflicto una vez
  quitada la pobreza y la región.

In [2]:
filas_exp2 = []
for ano in [2014, 2018, 2022]:
    sub = df_conf[df_conf['ano'] == ano]
    w = sub['peso_muestral'].values
    y = sub['pct_izquierda'].values
    region_dummies = pd.get_dummies(sub['region_dane'], prefix='reg', drop_first=True).astype(float)
    control = pd.concat([sub[['nbi_total']], region_dummies], axis=1).astype(float)
    per_ocu = sub['per_ocu'].values.astype(float)

    r2_sin = m.r2_ponderado(control.values, y, w)
    Xb = np.column_stack([control.values, per_ocu])
    r2_con = m.r2_ponderado(Xb, y, w)
    corr_parcial, resid_target, _ = m.correlacion_parcial_ponderada(y, control, per_ocu, w)

    es_zona = sub['departamento'].isin(m.ZONA_CONFLICTO_DEPARTAMENTOS).values
    gap_controlado = (np.average(resid_target[es_zona], weights=w[es_zona])
                       - np.average(resid_target[~es_zona], weights=w[~es_zona]))

    filas_exp2.append({
        'ano': ano, 'n': len(sub),
        'R2_NBI_region_SIN_per_ocu': r2_sin,
        'R2_NBI_region_CON_per_ocu': r2_con,
        'delta_R2_per_ocu': r2_con - r2_sin,
        'corr_parcial_per_ocu': corr_parcial,
        'gap_zona_conflicto_controlado': gap_controlado,
    })

tabla_exp2 = pd.DataFrame(filas_exp2)
tabla_exp2.round(4)

,ano,n,R2_NBI_region_SIN_per_ocu,R2_NBI_region_CON_per_ocu,delta_R2_per_ocu,corr_parcial_per_ocu,gap_zona_conflicto_controlado
0,2014,1118,0.0794,0.0796,0.0002,-0.0164,0.6627
1,2018,1119,0.4771,0.4831,0.0059,-0.1065,1.6618
2,2022,1119,0.6266,0.6282,0.0015,-0.0641,1.0177


**Lectura**: `per_ocu` no añade capacidad explicativa por encima de NBI+región en ningún año
(ΔR² entre 0,0002 y 0,006 — ruido). Su correlación parcial con el voto de izquierda es **negativa**
en los tres años, dirección contraria a la hipótesis de captura territorial. La brecha
zona_conflicto vs. resto, que en crudo llegaba a +13/+38 puntos (ver Experimento 1, bitácora §6),
se queda en menos de 2 puntos al controlar por NBI+región. Esto es lo que sostiene la decisión de
retirar la hipótesis de coacción electoral del diferenciador del proyecto.

## 2. Comprobación de robustez del hallazgo lateral — "¿se estructuró crecientemente el voto?"

Se recalcula el R²(NBI+región) para los 5 años (2006-2022) con el mismo procedimiento exacto
(regresión ponderada in-sample, filtrado por confiabilidad), junto con la descomposición
NBI-solo / región-sola, y la media/std del voto de izquierda de cada año.

In [3]:
filas_robustez = []
for ano in [2006, 2010, 2014, 2018, 2022]:
    sub = df_conf[df_conf['ano'] == ano]
    w = sub['peso_muestral'].values
    y = sub['pct_izquierda'].values
    media = np.average(y, weights=w)
    std = np.sqrt(np.average((y - media) ** 2, weights=w))

    region_dummies = pd.get_dummies(sub['region_dane'], prefix='reg', drop_first=True).astype(float)
    nbi = sub[['nbi_total']].astype(float)
    Xboth = pd.concat([nbi, region_dummies], axis=1).values

    r2_both = m.r2_ponderado(Xboth, y, w)
    r2_nbi = m.r2_ponderado(nbi.values, y, w)
    r2_reg = m.r2_ponderado(region_dummies.values, y, w)

    filas_robustez.append({
        'ano': ano, 'n': len(sub), 'media_voto_izq': media, 'std_voto_izq': std,
        'R2_NBI_mas_region': r2_both, 'R2_NBI_solo': r2_nbi, 'R2_region_sola': r2_reg,
        'pct_region_del_total': r2_reg / r2_both,
    })

tabla_robustez = pd.DataFrame(filas_robustez)
tabla_robustez.round(4)

,ano,n,media_voto_izq,std_voto_izq,R2_NBI_mas_region,R2_NBI_solo,R2_region_sola,pct_region_del_total
0,2006,1108,19.0638,15.2377,0.2484,0.0083,0.2457,0.9888
1,2010,1120,7.1788,8.1761,0.2972,0.0193,0.2853,0.9600
2,2014,1118,10.5035,7.3253,0.0794,0.0416,0.0466,0.5868
3,2018,1119,24.3157,18.8289,0.4771,0.1437,0.4644,0.9732
4,2022,1119,35.4405,22.4778,0.6266,0.1581,0.6131,0.9784


In [4]:
# Comprobacion 1: monotonia
diffs = tabla_robustez['R2_NBI_mas_region'].diff().dropna().values
print('Diferencias consecutivas de R2 (orden temporal):', diffs.round(4))
print('Es monotona creciente:', bool((diffs >= 0).all()))

# Comprobacion 2: confusion con la varianza disponible cada anio
corr_std_r2 = np.corrcoef(tabla_robustez['std_voto_izq'], tabla_robustez['R2_NBI_mas_region'])[0, 1]
corr_media_r2 = np.corrcoef(tabla_robustez['media_voto_izq'], tabla_robustez['R2_NBI_mas_region'])[0, 1]
print(f'\nCorrelacion std(voto) vs R2(NBI+region): {corr_std_r2:.4f}')
print(f'Correlacion media(voto) vs R2(NBI+region): {corr_media_r2:.4f}')

# Comprobacion 4: decisivo, cuanto es region vs NBI
print('\n% del R2 total atribuible a region sola, por anio:')
print(tabla_robustez[['ano', 'pct_region_del_total']].round(3).to_string(index=False))

Diferencias consecutivas de R2 (orden temporal): [ 0.0488 -0.2179  0.3978  0.1495]
Es monotona creciente: False

Correlacion std(voto) vs R2(NBI+region): 0.8872
Correlacion media(voto) vs R2(NBI+region): 0.8528

% del R2 total atribuible a region sola, por anio:
 ano  pct_region_del_total
2006                 0.989
2010                 0.960
2014                 0.587
2018                 0.973
2022                 0.978


**Lectura, las cuatro comprobaciones:**

- **Comprobación 1 (monotonía)**: NO monótona — hay una caída fuerte en 2014 (0,079) entre dos
  años con R² ya moderado (2006: 0,248, 2010: 0,297). El relato de "estructuración creciente" no
  se sostiene como tendencia limpia de 5 puntos.
- **Comprobación 2 (confusión con varianza disponible)**: correlación std(voto)-R² = 0,887 —
  muy alta. Los años con más voto de izquierda que explicar son mecánicamente los de R² más alto;
  gran parte del "salto" 2014→2022 es varianza disponible, no estructuración creciente real.
- **Comprobación 3 (confiabilidad/peso)**: ya aplicada en todo este notebook (filtro
  `baja_confiabilidad_electoral=0` desde la celda 2) — robusto, sin necesidad de comparación
  adicional (ver bitácora §8 para la comparación explícita con/sin filtro, diferencias en la 4ª
  cifra decimal).
- **Comprobación 4 (¿NBI o región?)**: decisiva — región sola explica 96-99% del R² combinado en
  4 de los 5 años (la excepción es 2014, con 58,7%, el mismo año atípico de las comprobaciones 1 y
  2). Si algo de esto se usa, el hallazgo correcto es "el voto se regionaliza", no "el voto se
  estructura crecientemente por pobreza".

**Decisión (ya tomada, aquí solo se reproduce el respaldo)**: el hallazgo lateral no se adopta
como nuevo eje del proyecto (Camino A se mantiene, no Camino B). Región, no pobreza creciente, es
lo que explica la mayor parte de la varianza disponible cuando la hay.

## 3. Cierre

Los números de esta celda y las anteriores son los que la bitácora de Fase 3 (§7, §8) y la memoria
final citan para sostener: (a) que la hipótesis de coacción electoral vía conflicto armado fue
puesta a prueba y refutada con evidencia, y (b) que la regionalización del voto explica lo que a
primera vista parecía "estructuración creciente por pobreza". Ambos ya viven en código ejecutable
de este repositorio, no solo en la bitácora interna.